# infviz on a Colab GPU

This notebook does two things on a Colab GPU:

1. **The speculation sweep** (the main reason to run it). It times speculative decoding against plain decoding for k = 0…8 at batch sizes 1 and 6, and writes one results file. It measures how this engine's speculation behaves on a GPU VM. (The L4 results in `results/sweep/` were produced with the Colab CLI rather than this notebook.)
2. **Recording a trace** (optional). It records a heavier run that you can replay in the web UI.

You need a **GPU runtime**: go to **Runtime → Change runtime type** and pick T4, L4 or A100.

In [ ]:
# Confirm that a GPU is attached
!nvidia-smi --query-gpu=name,memory.total,compute_cap --format=csv

## 1. Upload the engine code

The GitHub repository is private, so upload the engine as a zip. On your own machine, from the repo root:

```
zip -r engine.zip engine -x 'engine/.venv/*' '*/__pycache__/*'
```

Then run the cell below and pick `engine.zip`.

In [ ]:
import os
from google.colab import files
if not os.path.exists('/content/engine.zip'):  # already uploaded (e.g. with the Colab CLI): skip the picker
    files.upload()  # pick engine.zip

# engine.zip contains a top-level engine/ folder, so it unpacks to /content/engine
!unzip -q -o engine.zip -d /content/
%cd /content/engine

## 2. Install infviz

Torch is already installed on Colab, so this only adds the remaining dependencies and the `infviz` command.

In [ ]:
!pip install -q -e .

## 3. The speculation sweep

Every configuration serves the same 16 requests, all arriving at once, with KV pools large enough that nothing is evicted. Repeats are interleaved, and every (batch, k) pair is warmed up first and the warm-up discarded. Each row of the output records wall-clock time, time spent in draft and verify passes, tokens per second, tokens per verify pass and acceptance.

The dtype is chosen from the GPU: bf16 on compute capability 8.0 and newer (L4, A100), fp16 on older cards (T4 has no native bf16). Expect about 10–20 minutes on a T4.

In [ ]:
import torch, re

cc = torch.cuda.get_device_capability()
DTYPE = "bf16" if cc[0] >= 8 else "fp16"
GPU = re.sub(r"[^a-z0-9]+", "-", torch.cuda.get_device_name().lower()).strip("-")
SWEEP_OUT = f"/content/cuda-{GPU}-{DTYPE}.jsonl"

KS = "0,1,2,3,4,6,8"   # must start with 0, the plain-decoding baseline
BATCHES = "1,6"
REPEATS = 3
print(GPU, DTYPE, "->", SWEEP_OUT)

In [ ]:
!rm -f {SWEEP_OUT}
!infviz sweep --dtype {DTYPE} --ks {KS} --batches {BATCHES} --repeats {REPEATS} --out {SWEEP_OUT}

Check the run before downloading: every row should show `evict 0 preempt 0`. `match` counts the requests whose output is identical to plain greedy decoding; in fp16 or bf16 a few can differ through rounding near ties, which the results record rather than hide.

In [ ]:
import json, statistics as st
rows = [json.loads(l) for l in open(SWEEP_OUT)]
assert all(r["evictions"] == 0 and r["preemptions"] == 0 for r in rows), "a run hit KV pressure"
for b in sorted({r["batch"] for r in rows}):
    base = st.median(r["tok_per_s"] for r in rows if r["batch"] == b and r["k"] == 0)
    for k in sorted({r["k"] for r in rows}):
        sel = [r for r in rows if r["batch"] == b and r["k"] == k]
        tps = st.median(r["tok_per_s"] for r in sel)
        print(f"batch {b}  k {k}:  {tps:6.1f} tok/s   speed-up {tps / base:4.2f}x   "
              f"tok/pass {st.median(r['tok_per_pass'] for r in sel):4.2f}")

In [ ]:
from google.colab import files
try:
    files.download(SWEEP_OUT)   # commit it to results/sweep/ in the repo
except Exception as e:  # no browser attached (headless run): the file stays on the VM
    print("download unavailable here; the file is at", SWEEP_OUT)

## 4. Optional: record a trace

This runs a heavier workload headless, writes every engine event to a JSONL trace and checks that the trace reproduces the engine's own snapshots.

In [ ]:
# Target model (the model being served). The draft model proposes tokens for it.
TARGET = "Qwen/Qwen2.5-1.5B-Instruct"
# TARGET = "Qwen/Qwen2.5-7B-Instruct"   # needs an L4 or A100; shares the Qwen2.5 tokenizer with the draft

DRAFT = "Qwen/Qwen2.5-0.5B-Instruct"    # use "none" to turn off speculative decoding

N_REQUESTS = 32      # number of chat requests to serve
MEAN_GAP = 2.0       # mean number of engine steps between request arrivals
BLOCKS = 64          # target KV cache size, in blocks
MAX_BATCH = 8        # max requests running at once
SPEC_K = 4           # draft tokens proposed per step
OUT = "/content/colab_trace.jsonl"

In [ ]:
!infviz record --target {TARGET} --draft {DRAFT} --dtype {DTYPE} --blocks {BLOCKS} --max-batch {MAX_BATCH} --spec-k {SPEC_K} -n {N_REQUESTS} --mean-gap {MEAN_GAP} --seed 0 --out {OUT}

In [ ]:
from google.colab import files
try:
    files.download(OUT)
except Exception as e:  # no browser attached (headless run): the file stays on the VM
    print("download unavailable here; the file is at", OUT)

## 5. View the trace in the web UI

1. On your machine, start the web UI: `cd web && npm run dev`, then open the local address Vite prints.
2. Drag the downloaded `.jsonl` file onto the page to replay the run.

Alternatively, copy the file into `web/public/` and open the page with `?trace=colab_trace.jsonl`.